# 02 — 自定义 Deep Agents

**来源:** [LangChain Docs — Deep Agents / Customize](https://docs.langchain.com/oss/python/deepagents/customization)

Deep Agents 提供 `create_deep_agent()` 函数，通过丰富的参数让你为特定用例构建生产级的 Agent harness。

本笔记涵盖：
- `create_deep_agent` 完整参数签名
- 各参数的作用与用法
- Provider 集成示例（OpenAI / Anthropic / Azure / Google / Bedrock / HuggingFace）
- model 字符串 vs 实例化模型
- 三种传模型的方式（字符串、`init_chat_model`、直接类）

## 1. 完整函数签名

```python
create_deep_agent(
    model: str | BaseChatModel | None = None,
    tools: Sequence[BaseTool | Callable | dict[str, Any]] | None = None,
    *,
    system_prompt: str | SystemMessage | None = None,
    middleware: Sequence[AgentMiddleware] = (),
    subagents: Sequence[SubAgent | CompiledSubAgent | AsyncSubAgent] | None = None,
    skills: list[str] | None = None,
    memory: list[str] | None = None,
    permissions: list[FilesystemPermission] | None = None,
    backend: BackendProtocol | BackendFactory | None = None,
    interrupt_on: dict[str, bool | InterruptOnConfig] | None = None,
    response_format: ResponseFormat[ResponseT] | type[ResponseT] | dict[str, Any] | None = None,
    state_schema: type[DeepAgentState] | None = None,
    context_schema: type[ContextT] | None = None,
    checkpointer: Checkpointer | None = None,
    store: BaseStore | None = None,
    debug: bool = False,
    name: str | None = None,
    cache: BaseCache | None = None
) -> CompiledStateGraph[AgentState[ResponseT], ContextT, _InputAgentState, _OutputAgentState[ResponseT]]
```

## 2. 核心参数一览

| 参数 | 作用 |
|------|------|
| `model` | 选择模型（字符串或实例） |
| `system_prompt` | 自定义指令 |
| `tools` | Agent 可调用的领域工具 |
| `memory` | 启动时加载的 `AGENTS.md` 文件路径列表 |
| `skills` | Skills 目录，按需加载知识 |
| `backend` | 文件系统后端（默认 `StateBackend`） |
| `permissions` | 文件系统的路径级访问控制 |
| `subagents` | 用于委派任务的子 Agent |
| `middleware` | 额外中间件 |
| `interrupt_on` | 在工具调用前暂停等待人工审批 |
| `response_format` | 结构化输出 schema |
| `state_schema` | 自定义图状态 schema |
| `checkpointer` | 检查点器（持久化对话） |
| `store` | LangGraph 持久化存储 |
| `debug` | 是否开启调试模式 |
| `name` | Agent 名称 |
| `cache` | 缓存后端（如 `InMemoryCache`） |

## 3. 基本用法

最简单的 `create_deep_agent` 调用只需指定模型：

In [ ]:
from deepagents import create_deep_agent
from langchain.tools import tool

# 定义一个简单的工具
@tool
def get_time() -> str:
    """获取当前时间"""
    from datetime import datetime
from dotenv import load_dotenv
load_dotenv()
    return datetime.now().strftime("%H:%M:%S")

# 创建带自定义 prompt 和工具的 Agent
agent = create_deep_agent(
    model="deepseek-v4-flash",
    system_prompt="你是一个乐于助人的智能助手.",
    tools=[get_time],
    memory=["./AGENTS.md"],
    skills=["./skills/"],
)

print("Agent 创建成功（占位符）。")
print(f"模型: anthropic:claude-sonnet-4-6")
print(f"工具数: {len(agent.tools) if hasattr(agent, 'tools') else 'N/A'}")


Agent 创建成功（占位符）。
模型: anthropic:claude-sonnet-4-6
工具数: N/A


## 4. 模型传递方式

Deep Agents 支持三种方式指定模型：

### 4.1 方式一：字符串（`provider:model` 格式）

使用 `provider:model` 字符串，框架自动调用 `init_chat_model` 初始化：

In [ ]:
import os
from deepagents import create_deep_agent

# os.environ["OPENAI_API_KEY"] = "sk-..."
# agent = create_deep_agent(model="deepseek-v4-flash")

# 使用 DeepSeek 作为示例（当前环境已配置）
agent = create_deep_agent(model="deepseek:deepseek-v4-flash")
print(f"Agent created with model string: deepseek:deepseek-v4-flash")

Agent created with model string: deepseek:deepseek-v4-flash


### 4.2 方式二：`init_chat_model` 实例

先用 `init_chat_model` 配置好参数再传入：

In [ ]:
from langchain.chat_models import init_chat_model
from deepagents import create_deep_agent

# os.environ["OPENAI_API_KEY"] = "sk-..."
# model = init_chat_model(model="deepseek-v4-flash")
# agent = create_deep_agent(model=model)

# 使用 DeepSeek
model = init_chat_model(model="deepseek:deepseek-v4-flash")
agent = create_deep_agent(model=model)
print(f"Agent created with init_chat_model instance.")

Agent created with init_chat_model instance.


### 4.3 方式三：直接实例化 Provider 类

直接使用具体的 Chat Model 类（如 `ChatOpenAI`）：

In [ ]:
from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

# os.environ["OPENAI_API_KEY"] = "sk-..."
# model = ChatOpenAI(model="deepseek-v4-flash")
# agent = create_deep_agent(model=model)

# 使用本地 DeepSeek 兼容的 OpenAI 格式
# 省略 API 调用，仅展示模式
print("方式三：直接使用 Chat<Provider> 类")
print("适用场景：需要精细控制模型参数时")

方式三：直接使用 Chat<Provider> 类
适用场景：需要精细控制模型参数时


## 5. Provider 集成示例

以下展示各主流 Provider 的集成方式，均包含三种传模型的方法。

### 5.1 OpenAI

```bash
pip install -U "langchain[openai]"
```

In [ ]:
import os
from deepagents import create_deep_agent

# os.environ["OPENAI_API_KEY"] = "sk-..."
# 方式一：字符串
# agent = create_deep_agent(model="deepseek-v4-flash")

# 方式二：init_chat_model
# from langchain.chat_models import init_chat_model
# model = init_chat_model(model="deepseek-v4-flash")
# agent = create_deep_agent(model=model)

# 方式三：ChatOpenAI 类
# from langchain_openai import ChatOpenAI
# model = ChatOpenAI(model="deepseek-v4-flash")
# agent = create_deep_agent(model=model)

print("OpenAI 集成模式（三种方式均已注释，需配置 API Key）")

OpenAI 集成模式（三种方式均已注释，需配置 API Key）


### 5.2 Anthropic

```bash
pip install -U "langchain[anthropic]"
```

In [ ]:
import os
from deepagents import create_deep_agent

# os.environ["ANTHROPIC_API_KEY"] = "sk-..."
# 方式一：字符串
# agent = create_deep_agent(model="deepseek-v4-flash")

# 方式二：init_chat_model
# from langchain.chat_models import init_chat_model
# model = init_chat_model(model="deepseek-v4-flash")
# agent = create_deep_agent(model=model)

# 方式三：ChatAnthropic 类
# from langchain_anthropic import ChatAnthropic
# model = ChatAnthropic(model="deepseek-v4-flash")
# agent = create_deep_agent(model=model)

print("Anthropic 集成模式（三种方式均已注释，需配置 API Key）")

Anthropic 集成模式（三种方式均已注释，需配置 API Key）


### 5.3 Azure OpenAI

```bash
pip install -U "langchain[openai]"
```

In [ ]:
import os
from deepagents import create_deep_agent

# os.environ["AZURE_OPENAI_API_KEY"] = "..."
# os.environ["AZURE_OPENAI_ENDPOINT"] = "..."
# os.environ["OPENAI_API_VERSION"] = "2025-03-01-preview"

# 方式一：字符串
# agent = create_deep_agent(model="deepseek-v4-flash")

# 方式二：init_chat_model
# from langchain.chat_models import init_chat_model
# model = init_chat_model(
#     model="deepseek-v4-flash",
#     azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
# )
# agent = create_deep_agent(model=model)

# 方式三：AzureChatOpenAI 类
# from langchain_openai import AzureChatOpenAI
# model = AzureChatOpenAI(
#     model="deepseek-v4-flash",
#     azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
# )
# agent = create_deep_agent(model=model)

print("Azure OpenAI 集成模式（需配置 Azure 相关环境变量）")

Azure OpenAI 集成模式（需配置 Azure 相关环境变量）


### 5.4 Google Gemini

```bash
pip install -U "langchain[google-genai]"
```

In [ ]:
import os
from deepagents import create_deep_agent

# os.environ["GOOGLE_API_KEY"] = "..."

# 方式一：字符串
# agent = create_deep_agent(model="deepseek-v4-flash")

# 方式二：init_chat_model
# from langchain.chat_models import init_chat_model
# model = init_chat_model(model="deepseek-v4-flash")
# agent = create_deep_agent(model=model)

# 方式三：ChatGoogleGenerativeAI 类
# from langchain_google_genai import ChatGoogleGenerativeAI
# model = ChatGoogleGenerativeAI(model="deepseek-v4-flash")
# agent = create_deep_agent(model=model)

print("Google Gemini 集成模式（需配置 GOOGLE_API_KEY）")

Google Gemini 集成模式（需配置 GOOGLE_API_KEY）


### 5.5 AWS Bedrock

```bash
pip install -U "langchain[aws]"
```

In [ ]:
from deepagents import create_deep_agent

# 需先配置 AWS 凭证：
# https://docs.aws.amazon.com/bedrock/latest/userguide/getting-started.html

# 方式一：字符串 + model_provider 参数
# agent = create_deep_agent(
#     model="deepseek-v4-flash",
#     model_provider="bedrock_converse",
# )

# 方式二：init_chat_model
# from langchain.chat_models import init_chat_model
# model = init_chat_model(
#     model="deepseek-v4-flash",
#     model_provider="bedrock_converse",
# )
# agent = create_deep_agent(model=model)

# 方式三：ChatBedrock 类
# from langchain_aws import ChatBedrock
# model = ChatBedrock(model="deepseek-v4-flash")
# agent = create_deep_agent(model=model)

print("AWS Bedrock 集成模式（需配置 AWS 凭证）")

AWS Bedrock 集成模式（需配置 AWS 凭证）


### 5.6 HuggingFace

```bash
pip install -U "langchain[huggingface]"
```

In [ ]:
import os
from deepagents import create_deep_agent

# os.environ["HUGGINGFACEHUB_API_TOKEN"] = "hf_..."

# 方式一：字符串 + model_provider
# agent = create_deep_agent(
#     model="deepseek-v4-flash",
#     model_provider="huggingface",
#     temperature=0.7,
#     max_tokens=1024,
# )

# 方式二：init_chat_model
# from langchain.chat_models import init_chat_model
# model = init_chat_model(
#     model="deepseek-v4-flash",
#     model_provider="huggingface",
#     temperature=0.7,
#     max_tokens=1024,
# )
# agent = create_deep_agent(model=model)

# 方式三：ChatHuggingFace 类
# from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
# llm = HuggingFaceEndpoint(
#     repo_id="microsoft/Phi-3-mini-4k-instruct",
#     temperature=0.7,
#     max_length=1024,
# )
# model = ChatHuggingFace(llm=llm)
# agent = create_deep_agent(model=model)

print("HuggingFace 集成模式（需配置 HUGGINGFACEHUB_API_TOKEN）")

HuggingFace 集成模式（需配置 HUGGINGFACEHUB_API_TOKEN）


## 6. 高级自定义参数

除模型外，`create_deep_agent` 还支持以下高级参数用于复杂场景。

### 6.1 system_prompt — 自定义系统提示

设定 Agent 的行为和角色：

In [ ]:
from langchain_core.messages import SystemMessage
from deepagents import create_deep_agent

# 方式一：纯字符串
agent1 = create_deep_agent(
    model="deepseek:deepseek-v4-flash",
    system_prompt="你是一个中文翻译助手。",
)

# 方式二：SystemMessage 对象
agent2 = create_deep_agent(
    model="deepseek:deepseek-v4-flash",
    system_prompt=SystemMessage(content="你是一个 Python 代码助手。"),
)

print("两种 system_prompt 方式均支持。")

两种 system_prompt 方式均支持。


### 6.2 middleware — 中间件

中间件拦截每次模型调用，可实现日志、动态模型切换等：

In [ ]:
from langchain.agents.middleware import wrap_model_call
from deepagents import create_deep_agent

# 定义一个简单的日志中间件
@wrap_model_call
def log_middleware(request, handler):
    print(f"[日志] 模型调用: {request.model}")
    return handler(request)

agent = create_deep_agent(
    model="deepseek:deepseek-v4-flash",
    middleware=[log_middleware],
)

print("中间件已注入。")

中间件已注入。


### 6.3 interrupt_on — 人工审批

在特定工具调用前暂停，等待人工确认：

In [ ]:
from deepagents import create_deep_agent

# 拦截所有工具的调用，等待人工确认
agent = create_deep_agent(
    model="deepseek:deepseek-v4-flash",
    interrupt_on={"*": True},
)

# 或仅拦截特定工具
# agent = create_deep_agent(
#     model="deepseek:deepseek-v4-flash",
#     interrupt_on={"execute": True},
# )

print("HITL (Human-in-the-Loop) 模式已配置。")

HITL (Human-in-the-Loop) 模式已配置。


### 6.4 response_format — 结构化输出

强制模型输出遵循特定 schema：

In [ ]:
from typing import TypedDict
from langchain_core.pydantic_v1 import BaseModel, Field
from deepagents import create_deep_agent

# 使用 Pydantic 定义输出格式
class Joke(BaseModel):
    """一个笑话"""
    setup: str = Field(description="笑话的铺垫")
    punchline: str = Field(description="笑话的笑点")
    rating: int = Field(description="好笑程度 1-10")

# 注意：response_format 需要模型本身支持结构化输出
# agent = create_deep_agent(
#     model="deepseek:deepseek-v4-flash",
#     response_format=Joke,
# )

print("response_format 支持 Pydantic / TypedDict / dict。")

response_format 支持 Pydantic / TypedDict / dict。


---

**参考链接**
- [create_deep_agent API 参考](https://docs.langchain.com/oss/python/deepagents/customization)
- [配置 Harness 指南](https://docs.langchain.com/oss/python/deepagents/harness)
- [从零构建 Deep Agent 教程](https://docs.langchain.com/oss/python/deepagents/guides/build-from-scratch)
- [LangSmith 可观测性快速开始](https://docs.langchain.com/oss/python/observability)
- [Chat Model Integrations 列表](https://docs.langchain.com/oss/python/integrations/chat)